# AUC 改善实验台 v2.3- AutoCat（更多列当作类别特征）
在 v2.3 的基础上，新增 **Auto-Categorization（auto-cat）**：
- 自动将 **低基数/低唯一率/整数型** 的数值列视作**类别**；
- 支持两种方式：**替换原列** 或 **复制出 `<col>__as_cat` 新列**（保留数值原列）以便模型同时利用两种视角；
- 可配置参数：`max_card`、`max_ratio`、`integer_only`、`duplicate_numeric_as_cat`。

本笔记本将输出多份候选提交与全套诊断文件，便于只用一次线上机会时，挑选**最稳**的方案提交。

In [1]:
import os, json, gc, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool
from pandas.api.types import is_numeric_dtype
SEED=2025
np.random.seed(SEED)
SECONDS_PER_DAY=86400.0
NON_FEATURE_COLS=['id','label']
def to_dt(series):
    return pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', utc=True)


In [ ]:
# ===== 路径配置（按你要求） =====
import os
TRAIN_CSV = 'train/train.csv'
TRAIN_STMT_FEAT = 'train/train_statement_feature_v2.csv'
TEST_CSV = 'testaa/testaa.csv'                      # 如果没有可置为 None
TEST_STMT_FEAT = 'testaa/testaa_statement_feature_v2.csv'  # 如果没有可置为 None

OUT_SUBMISSION = 'output_Cat/submission.csv'
SAVE_OOF = 'output_Cat/oof_pred.csv'
SAVE_INFO = 'output_Cat/cv_info.json'

OUT_DIR = 'output_CAT/exp_v23_out'

# 创建目录（含输出目录与统一产物目录）
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUT_SUBMISSION), exist_ok=True)
os.makedirs(os.path.dirname(SAVE_OOF), exist_ok=True)
os.makedirs(os.path.dirname(SAVE_INFO), exist_ok=True)

print('路径OK：', TRAIN_CSV, TRAIN_STMT_FEAT, TEST_CSV, TEST_STMT_FEAT, OUT_DIR)

# ===== CatBoost 默认用 GPU 的轻量包装（自动回退 CPU） =====
from catboost import CatBoostClassifier as _CatBoostClassifier, Pool

def CatBoostClassifier(**params):
    """优先用 GPU（devices='0'），若不可用自动回退到 CPU。"""
    p = dict(params)
    p.setdefault('task_type', 'GPU')
    p.setdefault('devices', '0')
    try:
        m = _CatBoostClassifier(**p)
        # 试一次“空Pool”构造快速校验GPU是否可用；失败则回退
        _ = m.get_all_params()  # 触发加载（轻量）
        return m
    except Exception as e:
        print('[WARN] GPU 不可用，回退到 CPU：', e)
        p.pop('task_type', None); p.pop('devices', None)
        return _CatBoostClassifier(**p)

# 如果你的代码里有 params_base()/params_strong()，建议把 task_type/devices 缺省加上：
def _inject_gpu_default(d: dict) -> dict:
    dd = dict(d)
    dd.setdefault('task_type', 'GPU')
    dd.setdefault('devices', '0')
    return dd

# 如你在别处定义了 params_base/params_strong，可在创建模型前这样用：
# model = CatBoostClassifier(**_inject_gpu_default(params))


In [3]:
# 加载
train=pd.read_csv(TRAIN_CSV)
stmt=pd.read_csv(TRAIN_STMT_FEAT)
df=train.merge(stmt, on='id', how='left', suffixes=('', '_stmt'))
df['stmt_missing']=df['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in df.columns else 1
y=df['label'].astype(int)
X_all=df.drop(columns=['label']).copy()
X_test=None
if TEST_CSV is not None and os.path.exists(TEST_CSV):
    test=pd.read_csv(TEST_CSV)
    if TEST_STMT_FEAT is not None and os.path.exists(TEST_STMT_FEAT):
        stmt_t=pd.read_csv(TEST_STMT_FEAT)
        X_test=test.merge(stmt_t, on='id', how='left', suffixes=('', '_stmt'))
        X_test['stmt_missing']=X_test['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in X_test.columns else 1
    else:
        X_test=test.copy(); X_test['stmt_missing']=1
print('训练：', X_all.shape, '测试：', None if X_test is None else X_test.shape)

训练： (53480, 41) 测试： (20054, 41)


In [4]:
# 基础工程/填充/类型统一 + PSI + OOF TE（沿用 v2.3，增加 auto-cat）
def _coerce_numeric(series):
    return pd.to_numeric(series, errors='coerce')
def _safe_div(a,b):
    try: b=b.replace(0,np.nan)
    except Exception: b=np.where(b==0,np.nan,b)
    return a/b
def winsorize_df(X, lower=0.01, upper=0.99):
    Xw=X.copy()
    for c in Xw.columns:
        if c in NON_FEATURE_COLS: continue
        if is_numeric_dtype(Xw[c]):
            ql,qu=Xw[c].quantile(lower), Xw[c].quantile(upper)
            Xw[c]=Xw[c].clip(ql,qu)
    return Xw
def base_feature_engineering(df, do_winsor=False):
    if df is None: return None
    X=df.copy()
    for col in X.columns:
        if X[col].dtype=='O':
            X[col]=X[col].replace(['','',' ','nan','NaN','NULL','None'], np.nan)
    if 'level' in X.columns:
        X['level']=X['level'].astype(str)
        X['grade']=X['level'].str[0]
        X['subgrade']=pd.to_numeric(X['level'].str[1:], errors='coerce')
        X['grade_rank']=X['grade'].map({'A':5,'B':4,'C':3,'D':2,'E':1}).fillna(0).astype('Int64')
    if {'record_time','issue_time'}.issubset(X.columns):
        rec=to_dt(X['record_time']); iss=to_dt(X['issue_time'])
        X['days_issue_to_record']=((rec-iss).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    if {'record_time','history_time'}.issubset(X.columns):
        rec=to_dt(X['record_time']); his=to_dt(X['history_time'])
        X['days_history_to_record']=((rec-his).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    if {'balance','balance_limit'}.issubset(X.columns):
        X['balance_utilization']=_safe_div(_coerce_numeric(X['balance']), _coerce_numeric(X['balance_limit']))
        X['balance_utilization_sqrt']=np.sqrt(X['balance_utilization'].clip(lower=0))
    if {'balance_accounts','total_accounts'}.issubset(X.columns):
        X['acct_utilization']=_safe_div(_coerce_numeric(X['balance_accounts']), _coerce_numeric(X['total_accounts']))
    if {'loan','balance_limit'}.issubset(X.columns):
        X['loan_to_limit']=_safe_div(_coerce_numeric(X['loan']), _coerce_numeric(X['balance_limit']))
    if {'loan','term'}.issubset(X.columns):
        X['loan_per_month']=_safe_div(_coerce_numeric(X['loan']), _coerce_numeric(X['term']))
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade']:
        if c in X.columns:
            if c in ['level','grade']: X[c]=X[c].astype('object')
            else: X[c]=pd.to_numeric(X[c], errors='coerce').astype('Int64')
    if do_winsor: X=winsorize_df(X)
    for col in ['issue_time','record_time','history_time']:
        if col in X.columns: X.drop(columns=[col], inplace=True)
    return X
def get_cat_cols(df):
    cat_cols=[]
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade']:
        if c in df.columns: cat_cols.append(c)
    for c in df.select_dtypes(include='object').columns:
        if c not in cat_cols: cat_cols.append(c)
    cat_cols=[c for c in cat_cols if c not in NON_FEATURE_COLS and c in df.columns]
    return sorted(list(set(cat_cols)))
def fill_missing(df, cat_cols):
    X=df.copy()
    for c in X.columns:
        if c in NON_FEATURE_COLS: continue
        if c in cat_cols:
            X[c]=X[c].astype('object').fillna('Unknown'); continue
        if is_numeric_dtype(X[c]):
            X[c]=X[c].fillna(X[c].median()); continue
        xnum=pd.to_numeric(X[c], errors='coerce')
        if xnum.notna().any(): X[c]=xnum.fillna(xnum.median())
        else: X[c]=X[c].astype('object').fillna('Unknown')
    return X
def ensure_catboost_dtypes(df, cat_cols):
    X=df.copy()
    for c in cat_cols:
        if c in X.columns: X[c]=X[c].astype(str)
    for c in X.columns:
        if c in cat_cols or c in NON_FEATURE_COLS: continue
        X[c]=pd.to_numeric(X[c], errors='coerce').astype('float32')
    return X


In [5]:
# PSI 工具 & OOF Target Encoding（与 v2.3 一致）
def psi_numeric(train_s, test_s, bins=10):
    t=pd.to_numeric(train_s, errors='coerce'); q=t.quantile(np.linspace(0,1,bins+1)).values
    q=np.unique(q)
    if len(q)<=2: return 0.0
    t_bin=pd.cut(t, bins=q, include_lowest=True)
    s_bin=pd.cut(pd.to_numeric(test_s, errors='coerce'), bins=q, include_lowest=True)
    t_p=t_bin.value_counts(normalize=True); s_p=s_bin.value_counts(normalize=True)
    idx=t_p.index.union(s_p.index)
    t_p=t_p.reindex(idx).fillna(1e-6); s_p=s_p.reindex(idx).fillna(1e-6)
    return float(((t_p - s_p) * np.log(t_p / s_p)).sum())
def psi_categorical(train_s, test_s):
    t_p=train_s.astype(str).value_counts(normalize=True)
    s_p=test_s.astype(str).value_counts(normalize=True)
    idx=t_p.index.union(s_p.index)
    t_p=t_p.reindex(idx).fillna(1e-6); s_p=s_p.reindex(idx).fillna(1e-6)
    return float(((t_p - s_p) * np.log(t_p / s_p)).sum())
def compute_psi_report(X_train_full, X_test_full=None, record_time_train=None):
    if X_test_full is None:
        assert record_time_train is not None, '缺少 record_time.'
        dt=to_dt(record_time_train)
        cutoff=dt.quantile(0.5)
        A=X_train_full[dt<=cutoff]; B=X_train_full[dt>cutoff]
    else:
        A,B=X_train_full,X_test_full
    rows=[]
    for c in A.columns:
        if c in NON_FEATURE_COLS: continue
        try:
            psi=psi_numeric(A[c],B[c]) if is_numeric_dtype(A[c]) else psi_categorical(A[c],B[c])
        except Exception: psi=np.nan
        rows.append({'feature':c,'psi':psi})
    return pd.DataFrame(rows).sort_values('psi', ascending=False)
def oof_target_encoding(X, y, cat_cols, folds, smoothing=20, noise=0.01):
    X=X.copy(); global_mean=y.mean()
    te_maps={}; oof=pd.DataFrame(index=X.index)
    for c in cat_cols: oof[f'TE_{c}']=np.nan
    for tr_idx,va_idx in folds:
        y_tr=y.iloc[tr_idx]
        for c in cat_cols:
            s=X.iloc[tr_idx][c].astype(str)
            stats=s.to_frame().join(y_tr).groupby(c)['label'].agg(['mean','count']).rename(columns={'mean':'m','count':'n'})
            stats['te']=(stats['m']*stats['n']+global_mean*smoothing)/(stats['n']+smoothing)
            mp=stats['te']
            vals=X.iloc[va_idx][c].astype(str).map(mp).fillna(global_mean).values
            if noise>0: vals=vals*(1+noise*np.random.randn(len(vals)))
            oof.iloc[va_idx, oof.columns.get_loc(f'TE_{c}')]=vals
            te_maps[c]=mp.to_dict()
    full_maps={}
    for c in cat_cols:
        s=X[c].astype(str)
        stats=s.to_frame().join(y).groupby(c)['label'].agg(['mean','count']).rename(columns={'mean':'m','count':'n'})
        stats['te']=(stats['m']*stats['n']+global_mean*smoothing)/(stats['n']+smoothing)
        full_maps[c]=stats['te'].to_dict()
    return oof, full_maps


In [6]:
# Auto-Categorization（核心）：识别“数值但更像类别”的列
def detect_auto_cat_candidates(X, max_card=64, max_ratio=0.02, integer_only=True, exclude=None):
    if exclude is None: 
        exclude = set()
    cand = []
    n = len(X)
    for c in X.columns:
        if c in NON_FEATURE_COLS or c in exclude:
            continue
        # 仅考虑数值列
        if not is_numeric_dtype(X[c]):
            continue
        # 唯一值阈值
        nunq = X[c].nunique(dropna=True)
        uniq_ratio = nunq / max(1, n)
        if not (nunq <= max_card or uniq_ratio <= max_ratio):
            continue
        # 若只允许整数型，进行鲁棒的“近似整数”判断
        if integer_only:
            try:
                s = pd.to_numeric(X[c], errors='coerce')
                s = s[~pd.isna(s)]
                if len(s) == 0:
                    continue
                arr = s.to_numpy(dtype='float64', copy=False)
                # 允许微小浮点误差
                if np.all(np.isclose(arr, np.round(arr), rtol=0.0, atol=1e-8)):
                    cand.append(c)
            except Exception:
                # 出现任何类型问题，跳过该列，避免报错中断
                continue
        else:
            cand.append(c)
    return sorted(list(set(cand)))


In [7]:
# 统一准备：特征工程 -> PSI离散化 -> AutoCat -> 缺失 -> 类型 -> OOF TE（可选）
def prepare_matrix_with_autocat(df, y=None, record_time=None, X_test=None, drop_cols=None, do_winsor=False,
                                psi_bin_threshold=0.15,
                                autocat=True, max_card=64, max_ratio=0.02, integer_only=True,
                                duplicate_numeric_as_cat=True,
                                te=True, te_cardinality=(5,200), te_cv='skf', n_splits=5, seed=SEED):
    X=base_feature_engineering(df, do_winsor=do_winsor)
    XT=base_feature_engineering(X_test, do_winsor=do_winsor) if X_test is not None else None
    # drop cols
    if drop_cols:
        dd=[c for c in drop_cols if c in X.columns]
        if dd: X=X.drop(columns=dd)
        if XT is not None:
            ddt=[c for c in drop_cols if c in XT.columns]
            if ddt: XT=XT.drop(columns=ddt)
    # PSI 数值分箱
    psi_rep=compute_psi_report(X, XT, record_time_train=record_time)
    psi_rep.to_csv(os.path.join(OUT_DIR, 'psi_report_autocat.csv'), index=False)
    high_psi_num=[]
    for c in X.columns:
        if c in NON_FEATURE_COLS: continue
        if is_numeric_dtype(X[c]):
            row=psi_rep.loc[psi_rep['feature']==c]
            if not row.empty and float(row['psi'].values[0])>=psi_bin_threshold:
                high_psi_num.append(c)
    bin_edges={}
    for c in high_psi_num:
        try:
            q=np.unique(X[c].quantile(np.linspace(0,1,11)).values)
            if len(q)<=2: continue
            X[f'{c}_bin']=pd.cut(X[c], bins=q, include_lowest=True).astype(str)
            if XT is not None:
                XT[f'{c}_bin']=pd.cut(XT[c], bins=q, include_lowest=True).astype(str)
            bin_edges[c]=q.tolist()
            # 保留原数值列不删除，给 auto-cat 或模型做选择
        except Exception:
            pass
    with open(os.path.join(OUT_DIR, 'bin_edges_autocat.json'),'w',encoding='utf-8') as f:
        json.dump(bin_edges, f, ensure_ascii=False, indent=2)

    # AutoCat：识别数值中的类别候选
    extra_cat=[]
    if autocat:
        exclude=set(high_psi_num)  # 高漂移数值已分箱，优先让 *_bin 当类别
        cand=detect_auto_cat_candidates(X, max_card=max_card, max_ratio=max_ratio, integer_only=integer_only, exclude=exclude)
        for c in cand:
            if duplicate_numeric_as_cat:
                newc=f'{c}__as_cat'
                X[newc]=X[c]
                if XT is not None and c in XT.columns:
                    XT[newc]=XT[c]
                extra_cat.append(newc)
            else:
                extra_cat.append(c)
    # 类别列集合（基础类别 + 分箱 + auto-cat 生成的 __as_cat）
    cat_cols=get_cat_cols(X)
    # 把 *_bin 和 extra_cat 都纳入类别
    for c in list(X.columns):
        if c.endswith('_bin') and c not in cat_cols: cat_cols.append(c)
    for c in extra_cat:
        if c not in cat_cols and c in X.columns: cat_cols.append(c)

    # 缺失 & 类型统一
    X=fill_missing(X, cat_cols)
    X=ensure_catboost_dtypes(X, cat_cols)
    if XT is not None:
        XT=fill_missing(XT, cat_cols)
        XT=ensure_catboost_dtypes(XT, cat_cols)

    # 目标编码（OOF），仅对中等基数类别
    te_cols=[]
    if te:
        for c in cat_cols:
            card=X[c].nunique()
            if te_cardinality[0]<=card<=te_cardinality[1]: te_cols.append(c)
    te_oof, te_maps = None, None
    if te and y is not None and len(te_cols)>0:
        if te_cv=='time':
            dt=to_dt(record_time)
            order=np.argsort(dt.values)
            idx=np.arange(len(dt))[order]
            chunks=np.array_split(idx, 5)
            folds=[(np.concatenate(chunks[:k]), chunks[k]) for k in range(1,5)]
        else:
            skf=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
            folds=list(skf.split(X, y))
        te_oof, te_maps=oof_target_encoding(pd.concat([X, y.rename('label')], axis=1), y, te_cols, folds)
        X=pd.concat([X, te_oof], axis=1)
        for c,mp in te_maps.items():
            pd.DataFrame(list(mp.items()), columns=[c, f'TE_{c}']).to_csv(os.path.join(OUT_DIR, f'te_map_{c}.csv'), index=False)

    # 记录 auto-cat 决策
    with open(os.path.join(OUT_DIR, 'autocat_selected.json'),'w',encoding='utf-8') as f:
        json.dump({'extra_cat': extra_cat, 'psi_binned': list(bin_edges.keys())}, f, ensure_ascii=False, indent=2)

    return X, XT, get_cat_cols(X), te_cols


In [8]:
# 训练/评估 & 参数
def run_catboost_cv(X, y, cat_cols, params, n_splits=5, seed=SEED, cv_type='skf', record_time=None):
    if cv_type=='skf':
        splitter=list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X,y))
    else:
        dt=to_dt(record_time)
        order=np.argsort(dt.values)
        idx=np.arange(len(dt))[order]
        chunks=np.array_split(idx, n_splits)
        splitter=[(np.concatenate(chunks[:k]), chunks[k]) for k in range(1,n_splits)]
    feat_cols=[c for c in X.columns if c not in NON_FEATURE_COLS]
    cat_idx=[feat_cols.index(c) for c in cat_cols if c in feat_cols]
    oof=np.zeros(len(X)); folds_auc=[]; best_iters=[]
    for k,(tr,va) in enumerate(splitter,1):
        X_tr,X_va=X.iloc[tr][feat_cols], X.iloc[va][feat_cols]
        y_tr,y_va=y.iloc[tr], y.iloc[va]
        model=CatBoostClassifier(**params)
        model.fit(Pool(X_tr, label=y_tr, cat_features=cat_idx), eval_set=Pool(X_va, label=y_va, cat_features=cat_idx), verbose=False)
        va_pred=model.predict_proba(X_va)[:,1]
        oof[va]=va_pred
        folds_auc.append(roc_auc_score(y_va, va_pred))
        best_iters.append(model.get_best_iteration())
    return oof, folds_auc, best_iters
def params_base(seed=SEED):
    return dict(loss_function='Logloss', eval_metric='AUC', iterations=8000, learning_rate=0.02, depth=6,
                l2_leaf_reg=10.0, random_seed=seed, bootstrap_type='Bayesian', bagging_temperature=0.5,
                rsm=0.8, random_strength=2.0, border_count=128, grow_policy='SymmetricTree', od_type='Iter', od_wait=400,
                verbose=False, allow_writing_files=False, auto_class_weights='Balanced')
def params_strong(seed=SEED):
    return dict(loss_function='Logloss', eval_metric='AUC', iterations=12000, learning_rate=0.015, depth=5,
                l2_leaf_reg=20.0, random_seed=seed, bootstrap_type='Bayesian', bagging_temperature=0.8,
                rsm=0.7, random_strength=3.0, border_count=128, grow_policy='SymmetricTree', od_type='Iter', od_wait=600,
                verbose=False, allow_writing_files=False, auto_class_weights='Balanced')


In [9]:
# 实验清单：探索 auto-cat 阈值 & 是否复制数值为类别
EXPS=[
  dict(name='AC1_dup_card64_ratio2', drop_cols=['zip_code','title','residence'], do_winsor=True,
       psi_bin_threshold=0.15, autocat=True, max_card=64, max_ratio=0.02, integer_only=True, duplicate_numeric_as_cat=True,
       te=True, te_cv='time', strong=False),
  dict(name='AC2_dup_card32_ratio1', drop_cols=['zip_code','title','residence'], do_winsor=True,
       psi_bin_threshold=0.15, autocat=True, max_card=32, max_ratio=0.01, integer_only=True, duplicate_numeric_as_cat=True,
       te=True, te_cv='time', strong=False),
  dict(name='AC3_replace_card64_ratio2', drop_cols=['zip_code','title','residence'], do_winsor=True,
       psi_bin_threshold=0.15, autocat=True, max_card=64, max_ratio=0.02, integer_only=True, duplicate_numeric_as_cat=False,
       te=True, te_cv='time', strong=True),
  dict(name='AC4_dup_card64_ratio2_noTE', drop_cols=['zip_code','title','residence'], do_winsor=True,
       psi_bin_threshold=0.15, autocat=True, max_card=64, max_ratio=0.02, integer_only=True, duplicate_numeric_as_cat=True,
       te=False, te_cv='time', strong=True),
]
pd.DataFrame(EXPS).to_csv(os.path.join(OUT_DIR,'experiment_configs_autocat.csv'), index=False)
print('实验方案：', [e['name'] for e in EXPS])


实验方案： ['AC1_dup_card64_ratio2', 'AC2_dup_card32_ratio1', 'AC3_replace_card64_ratio2', 'AC4_dup_card64_ratio2_noTE']


In [10]:
# 运行实验并产出文件
summary=[]
for exp in EXPS:
    name=exp['name']
    X_fe, XT_fe, cat_cols, te_cols = prepare_matrix_with_autocat(
        X_all, y=y, record_time=train['record_time'], X_test=X_test,
        drop_cols=exp['drop_cols'], do_winsor=exp['do_winsor'],
        psi_bin_threshold=exp['psi_bin_threshold'], autocat=exp['autocat'], max_card=exp['max_card'], max_ratio=exp['max_ratio'], integer_only=exp['integer_only'],
        duplicate_numeric_as_cat=exp['duplicate_numeric_as_cat'], te=exp['te'], te_cv=exp['te_cv']
    )
    feat_cols=[c for c in X_fe.columns if c not in NON_FEATURE_COLS]
    with open(os.path.join(OUT_DIR,f'features_{name}.txt'),'w',encoding='utf-8') as f:
        f.write('\n'.join(feat_cols))
    with open(os.path.join(OUT_DIR,f'te_cols_{name}.txt'),'w',encoding='utf-8') as f:
        f.write('\n'.join(te_cols))
    params = params_base(SEED) if not exp.get('strong', False) else params_strong(SEED)
    # 分层与时间两种验证
    oof_skf, folds_skf, it_skf = run_catboost_cv(X_fe, y, cat_cols, params, n_splits=5, seed=SEED, cv_type='skf', record_time=train['record_time'])
    auc_skf=float(roc_auc_score(y, oof_skf))
    pd.DataFrame({'id':X_all['id'],'oof_pred':oof_skf,'label':y}).to_csv(os.path.join(OUT_DIR,f'oof_{name}.csv'), index=False)

    oof_time, folds_time, it_time = run_catboost_cv(X_fe, y, cat_cols, params, n_splits=5, seed=SEED, cv_type='time', record_time=train['record_time'])
    auc_time=float(roc_auc_score(y, oof_time))
    pd.DataFrame({'id':X_all['id'],'oof_pred':oof_time,'label':y}).to_csv(os.path.join(OUT_DIR,f'oof_time_{name}.csv'), index=False)

    # 全量复训用于重要性与提交
    cat_idx=[feat_cols.index(c) for c in cat_cols if c in feat_cols]
    model=CatBoostClassifier(**{**params,'iterations':max(200,int(np.mean(it_skf)))})
    model.fit(Pool(X_fe[feat_cols], label=y, cat_features=cat_idx), verbose=False)
    imp = model.get_feature_importance(Pool(X_fe[feat_cols], label=y, cat_features=cat_idx), type='PredictionValuesChange')
    pd.DataFrame({'feature':feat_cols,'importance':imp}).sort_values('importance',ascending=False).to_csv(os.path.join(OUT_DIR,f'feature_importance_{name}.csv'), index=False)

    sub_path=None
    if XT_fe is not None:
        preds=model.predict_proba(XT_fe[feat_cols])[:,1]
        sub=pd.DataFrame({'id': X_test['id'].values, 'prob': preds})
        sub_path=os.path.join(OUT_DIR,f'submission_{name}.csv')
        sub.to_csv(sub_path, index=False)

    with open(os.path.join(OUT_DIR,f'cv_{name}.json'),'w',encoding='utf-8') as f:
        json.dump({'cv_auc_skf':auc_skf,'fold_auc_skf':[float(x) for x in folds_skf],'best_iter_skf':[int(x) for x in it_skf],
                   'cv_auc_time':auc_time,'fold_auc_time':[float(x) for x in folds_time],'best_iter_time':[int(x) for x in it_time]}, f, ensure_ascii=False, indent=2)

    summary.append(dict(name=name, cv_auc_skf=auc_skf, cv_auc_time=auc_time, features=len(feat_cols), te=exp['te'], strong=exp.get('strong',False),
                        autocat=exp['autocat'], max_card=exp['max_card'], max_ratio=exp['max_ratio'], duplicate_numeric_as_cat=exp['duplicate_numeric_as_cat'], submission=sub_path))
    print(f"完成 {name}: skf={auc_skf:.4f}, time={auc_time:.4f}, 提交={sub_path}")
    gc.collect()

summary_df=pd.DataFrame(summary)
summary_df.to_csv(os.path.join(OUT_DIR,'experiments_summary_autocat.csv'), index=False)
summary_df

KeyError: "['TE_career', 'TE_grade', 'TE_level', 'TE_active_days__as_cat', 'TE_balance_accounts__as_cat', 'TE_career__as_cat', 'TE_expense_count__as_cat', 'TE_income_count__as_cat', 'TE_loan__as_cat', 'TE_span_days__as_cat', 'TE_subgrade__as_cat', 'TE_total_accounts__as_cat', 'TE_tx_count__as_cat'] not in index"